# §55 — **K5dfiDoTA**: K5dfi's recipe trained on the recovered DoTA train clips, s2024

Rule: `core/docs/final/KIM_RUN.md` **§55**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. PRELIMINARY (n = 1). **Decides nothing; K5f stays the reference. K5dfiDoTA is in-domain on DoTA, never zero-shot.** The pixel gate was not valid; training is the user's decision.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-4 | setup, deps, helpers, preflight | no | 3-5 min |
| 5 | tests — **raises if red** | no | 1-2 min |
| 6 | extract: 3,020 clips -> CLIP + V2-S rows at stride 3 (resumable, re-run after a disconnect) | **yes** | about 1-2 h on T4 (2 cores), less on L4 / A100; two sessions can share it |
| 7 | corpus (12 rows, hop 6) + A2 statistics **fitted on DoTA train**, gate R2, restat of the four zero-shot bakes, KNN | no | 15-25 min |
| 8 | train K5dfiDoTA (1,764 steps) + guards | **yes** | as K5dfi |
| 9 | reads: DoTA-CAP-dev, DADA, TAD, DAD, Nexar val | yes | 40-60 min |
| 10 | table vs K5dfi -> **paste `REPORTS/k5dfidota_report.md` to Claude** | no | 2 min |

A re-run skips what is on Drive. Thesis / Thesis-V2 are only read; everything new goes to `Thesis-Final/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, Nexar caches, census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, frames, K5dfi / K5f and their reads; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 55. K5dfiDoTA' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §55 -- re-upload kat-vad/'
for tool in ('dota_train_extract', 'dota_train_build'):
    assert (REPO / f'core/tools/{tool}.py').is_file(), f'no core/tools/{tool}.py -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5       = constants.KIM_MAIN_ARM                  # "K5": names run / eval sub-folders only
CTRL     = constants.KIM_DVS_IGNORE_ROW            # K5dfi: the control
K5F      = constants.KIM_REFERENCE_ARM             # printed row
ROW      = constants.KIM_DOTA_TRAIN_ROW            # K5dfiDoTA
IGN      = constants.DVS_ANCHOR_MODE_IGNORE
SEED     = constants.KIM_SEED
ENC      = constants.V5_ENCODER                    # V2-S
OFF      = constants.V2_OFF
DATASET  = constants.NEXAR_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL         # nexar_test is never read here
TAD_NAME = constants.TAD_DATASET
DAD_NAME, DAD_STRIDE, DAD_SET = constants.DAD_DATASET, constants.DAD_STRIDE, constants.DAD_OUTPUT_SET

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'   # defs of every Nexar read
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'   # Nexar statistics: gate R2 only
CTRL_ROOT = constants.OUTPUT_ROOT / constants.KIM_DVS_IGNORE_OUTPUT_DIRNAME / CTRL
CTRL_RUN, CTRL_EV = CTRL_ROOT / K5 / f's{SEED}', CTRL_ROOT / 'eval'
K5F_EV   = constants.OUTPUT_ROOT / constants.KIM_FIXED_OUTPUT_DIRNAME / K5F / 'eval'
# zero-shot inputs: the bakes K5dfi was read on (Nexar statistics), restated to DoTA's below
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
A2_TAD   = V2_CACHE / 'v2' / 'A2_S' / f'{TAD_NAME}_s8_nexar'
A2_DAD   = constants.CACHE_ROOT / 'v2' / 'A2_S' / f'{DAD_NAME}_s{DAD_STRIDE}_nexar'
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
NEX, DST_SUF = constants.NEXAR_STATS_BAKE_SUFFIX, constants.DOTA_TRAIN_STATS_BAKE_SUFFIX
ZS_SRC   = {'dada': A2_DADA, 'dota': A2_DOTA, 'tad': A2_TAD, DAD_SET: A2_DAD}
assert all(p.name.endswith(NEX) for p in ZS_SRC.values())
ZS_DST   = {s: constants.CACHE_ROOT / 'v2' / 'A2_S' / (p.name[: -len(NEX)] + DST_SUF) for s, p in ZS_SRC.items()}
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
TAD      = constants.DATA_ROOT / TAD_NAME
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
# DoTA train: frames (read), features / corpus / bake (written)
ARCHIVE  = Path(DRIVE_F) / 'data' / constants.DOTA_TRAIN_FRAMES_DIRNAME
CLIP_S3  = constants.CLIP_CACHE_DIR / constants.DOTA_TRAIN_CLIP_DATASET
VIDEO_S3 = constants.VIDEO_CACHE_DIR / ENC / f'{constants.DOTA_TRAIN_VIDEO_DATASET}_squash'
DATA     = Path(DRIVE_F) / 'data' / constants.DOTA_TRAIN_CORPUS_DIRNAME
BAKE     = constants.CACHE_ROOT / 'v2' / 'A2_S' / constants.DOTA_TRAIN_BAKE_DIRNAME
KNN      = constants.CACHE_ROOT / 'knn' / DATA.name / constants.KNN_CACHE_FILENAME
ROOT     = constants.OUTPUT_ROOT / constants.KIM_DOTA_TRAIN_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
BUILD    = REPORTS / constants.DOTA_TRAIN_BUILD_REPORT
LOCAL    = Path('/content/k5dfidota')
LOCAL.mkdir(parents=True, exist_ok=True)


def row_paths(row):
    run_dir = ROOT / row / K5 / f's{SEED}'
    return run_dir, run_dir / 'checkpoint_last.pt', ROOT / row / 'eval'


import hashlib  # noqa: E402
import urllib.request  # noqa: E402

ANN = LOCAL / 'DoTA_annotations.zip'              # DoTA's official annotation zip (10 MB)
if not ANN.exists():
    urllib.request.urlretrieve('https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip', ANN)
frozen = json.loads((constants.DOTA_TRAIN_SPLITS_DIR / constants.DOTA_TRAIN_MAP_FILENAME).read_text())
assert hashlib.sha256(ANN.read_bytes()).hexdigest() == frozen['sources']['annotations_sha256'], 'annotation zip is not the one the map was built on'

INPUTS = (('frames manifest', ARCHIVE / constants.DOTA_CAP_FRAMES_MANIFEST), ('Nexar_window defs', WINDOW / constants.DEFS_FILENAME),
          ('Nexar A2 bake', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME),
          ('K5dfi checkpoint', CTRL_RUN / 'checkpoint_last.pt'), ('K5dfi verified', CTRL_RUN / 'verified.json'),
          ('K5dfi config', CTRL_RUN / 'config.yaml'), ('K5dfi metrics', CTRL_RUN / 'metrics.jsonl'),
          ('K5dfi nexar_val read', CTRL_EV / 'nexar' / CTRL / 'nexar_readout.json'),
          *[(f'{r} {s} read', ev / s / K5) for r, ev in ((CTRL, CTRL_EV), (K5F, K5F_EV)) for s in ('dota', 'dada', 'tad', DAD_SET)],
          ('DADA A2 bake', A2_DADA), ('DoTA-CAP A2 bake', A2_DOTA), ('TAD A2 bake', A2_TAD), ('DAD A2 bake', A2_DAD),
          ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT), ('DoTA s1', DOTA_S1),
          ('DoTA metadata', DOTA / 'metadata_val.json'), ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME),
          ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME),
          ('Nexar census', CENSUS), ('Nexar CLIP s1', NEXAR_X), ('Nexar V2-S s1', NEXAR_U))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | frames {ARCHIVE} | runs / reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 6, 8, 9 need one'
df -h /content | tail -1

In [ ]:
import numpy as np  # noqa: E402
import torch  # noqa: E402
import yaml  # noqa: E402

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.dota_train_extract import train_clips  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    names = [f.name for f in src.iterdir() if f.is_file()] if ids is None else \
        [f'{v}.npy' for v in ids] + [m.name for p in ('*.json', '*.npz') for m in src.glob(p)]
    todo = [n for n in names if not (dst / n).exists()]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda n: shutil.copy2(src / n, dst / n), todo))
    print(f'  staged {src.name}: {len(todo)} files in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def on_drive(folder, min_bytes=1024):
    """Ids of every finished ``.npy`` in ``folder`` from ONE listing (save_array = tmp + rename)."""
    if not folder.is_dir():
        return set()
    with os.scandir(folder) as entries:
        return {e.name[:-4] for e in entries if e.name.endswith('.npy') and e.stat().st_size >= min_bytes}


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(run_dir, steps):
    rec, ckpt = run_dir / 'verified.json', run_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


DEFAULTS = _flat(Config().to_dict())


def config_diff(base, got):
    """Keys that differ; a key added after ``base`` trained may sit at its default."""
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k):
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out


# Preflight on small files only (C39)
ctrl_cfg = _flat(yaml.safe_load((CTRL_RUN / 'config.yaml').read_text()))
assert all(ctrl_cfg[k] for k in ('kim.enabled', 'kim.prototypes', 'kim.edge_mask', 'kim.seam_mask')), 'CTRL_RUN is not K5dfi'
assert (ctrl_cfg['dvs.theta'], ctrl_cfg['loss.dvs_anchor_mode'], ctrl_cfg['data.is_egocentric'], ctrl_cfg['data.dataset']) == (0.7, IGN, False, DATASET), 'CTRL_RUN is not K5dfi'
CTRL_STEPS = json.loads((CTRL_RUN / 'verified.json').read_text())['step']
assert is_verified(CTRL_RUN, CTRL_STEPS) and metrics_rows(CTRL_RUN / 'metrics.jsonl')[-1]['global_step'] == CTRL_STEPS, 'K5dfi: J10'
SHA = manifest(A2_NEXAR)['train_ids_sha1']
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA), ('DoTA-CAP', A2_DOTA), ('TAD', A2_TAD), ('DAD', A2_DAD)):
    m = manifest(d)
    assert (m['crn'], m['motion'], m['train_ids_sha1']) == (OFF, ENC, SHA), f'{name} bake: {m}'
frames_manifest = json.loads((ARCHIVE / constants.DOTA_CAP_FRAMES_MANIFEST).read_text())
assert frames_manifest['pass'], 'the packed frames did not pass verify'
TRAIN = train_clips(constants.DOTA_TRAIN_SPLITS_DIR / constants.DOTA_TRAIN_MAP_FILENAME)
DADA_IDS, CAP_IDS = test_sources(T2), sorted(load_split(CAP_DEV))
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(TRAIN) == 3020 and not set(TRAIN) & set(CAP_IDS), len(TRAIN)
assert len(DADA_IDS) == 370 and len(TAD_IDS) == constants.V2_TADZ_VIDEOS and len(DAD_IDS) == constants.DAD_TEST_VIDEOS
print(f'preflight OK: K5dfi step {CTRL_STEPS} | DoTA train clips {len(TRAIN)} ({sum(c["dota_frames"] for c in TRAIN.values())} frames) | '
      f'DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)}, TAD {len(TAD_IDS)}, DAD {len(DAD_IDS)}, {NEXAR_SPLIT} {len(NEXAR_IDS)}')

## 5. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_dota_train_corpus.py',
     'core/tests/test_dota_train.py', 'core/tests/test_nexar_pipeline.py', 'core/tests/test_kim.py',
     'core/tests/test_dvs_dataset.py', 'core/tests/test_v2_restat.py'])

## 6. Extract CLIP + V2-S rows at stride 3 (GPU, resumable). **Re-run this cell after a disconnect.**

Clips are read and decoded in parallel (CPU cores + 4 threads); 400 clips per chunk, copied to Drive with 16 threads. The log prints the time left. JPEG decode is the limit, so a runtime with **more CPU cores** (L4 / A100) is faster than a T4.

**Two sessions at once:** set `SHARD = (0, 2)` in one and `SHARD = (1, 2)` in the other; when both are done, continue in **one** of them with `SHARD = (0, 1)` (it re-checks that everything is on Drive).

In [ ]:
SHARD = (0, 1)          # (this session, number of sessions)
CHUNK = 400
print(f'{os.cpu_count()} CPU cores')
t_start = time.time()
ids_all = sorted(TRAIN)
done = on_drive(CLIP_S3) & on_drive(VIDEO_S3)
print(f'on Drive: {len(done & set(ids_all))} / {len(ids_all)} clips have both caches')
mine = ids_all[SHARD[0]::SHARD[1]]


def publish_many(src, dst):
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda f: publish(f, dst / f.name), sorted(src.iterdir())))


for k in range(0, len(mine), CHUNK):
    todo = [v for v in mine[k:k + CHUNK] if v not in done]
    if not todo:
        continue
    assert torch.cuda.is_available(), 'no GPU: extraction on CPU would take days'
    print(f'=== chunk {k // CHUNK + 1} / {-(-len(mine) // CHUNK)}: {len(todo)} clips')
    tag = f'{SHARD[0]}of{SHARD[1]}_{k:04d}'
    work = LOCAL / 'extract' / tag
    run([sys.executable, '-m', 'core.tools.dota_train_extract', '--archive-dir', ARCHIVE,
         '--ids-file', write_ids(LOCAL / 'ids' / f'extract_{tag}.txt', todo), '--encoder', ENC,
         '--clip-out', work / 'x', '--video-out', work / 'u', '--report', REPORTS / 'extract' / f'extract_{tag}.json'],
        log=LOCAL / 'extract.log')
    publish_many(work / 'x', CLIP_S3)
    publish_many(work / 'u', VIDEO_S3)
    shutil.rmtree(work)
flush_drive()
done = on_drive(CLIP_S3) & on_drive(VIDEO_S3)
left_mine = [v for v in mine if v not in done]
assert not left_mine, f'{len(left_mine)} clips of this shard still to extract: re-run this cell'
left = [v for v in ids_all if v not in done]
print(f'extraction: {len(ids_all) - len(left)} / {len(ids_all)} ({(time.time() - t_start) / 60:.0f} min this session)')
assert not left, f'this shard is done; {len(left)} clips belong to the other session: wait, then re-run with SHARD = (0, 1)'

## 7. Corpus `DoTA_train_window` (12 rows, hop 6); A2 statistics **fitted on DoTA train**; gate R2; restat of the zero-shot bakes; KNN on plain CLIP

In [ ]:
from core.data.v2_inputs import load_stats  # noqa: E402
from core.data.v2_splits import lines_sha1   # noqa: E402

L_X = stage(CLIP_S3, LOCAL / 'stage' / 'x', ids_all)
L_U = stage(VIDEO_S3, LOCAL / 'stage' / 'u', ids_all)
vm = json.loads((L_U / constants.VIDEO_MANIFEST_FILENAME).read_text())
assert (vm['clip_frame_step'], vm['assumed_fps'], vm['stride']) == (constants.DOTA_VIDEOMAE_FRAME_STEP, constants.DOTA_FPS, constants.DOTA_TRAIN_STRIDE), vm
if not BUILD.exists():
    local = LOCAL / 'data'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.dota_train_build', '--annotations', ANN, '--clip-dir', L_X,
         '--out-dir', local, '--report', LOCAL / 'build_report.json'], log=LOCAL / 'build.log')
    publish(local, DATA)
    publish(LOCAL / 'build_report.json', BUILD)
build = json.loads(BUILD.read_text())
info, PLAN = build['construction'], build['plan']
assert (info['train_items'], info['train_abnormal'], info['train_normal'], info['clips_with_windows']) == (12937, 9386, 3551, 3019), f'not §55 DC2: {info}'
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']
assert (STEPS, EPOCHS) == (1764, 6), PLAN
ids_file = write_ids(LOCAL / 'ids' / 'bake.txt', ids_all)
windows = json.loads((DATA / constants.WINDOWS_FILENAME).read_text())
TRAIN_SRC = sorted({w['source'] for w in windows.values()})
assert len(TRAIN_SRC) == 3019 and set(TRAIN_SRC) <= set(ids_all), len(TRAIN_SRC)
fit_file, SHA_DOTA = write_ids(LOCAL / 'ids' / 'fit.txt', TRAIN_SRC), lines_sha1(TRAIN_SRC)
if not ((BAKE / constants.V2_INPUT_MANIFEST_FILENAME).exists() and set(ids_all) <= on_drive(BAKE)):
    local = LOCAL / 'bake'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'fit-ids', '--train-ids-file', fit_file, '--ids-file', ids_file,
         '--clip-dir', L_X, '--video-dir', L_U, '--stride', 1, '--crn', OFF, '--motion', ENC, '--out-dir', local], log=LOCAL / 'bake.log')
    publish(local, BAKE)
    flush_drive()
m = manifest(BAKE)
assert (m['crn'], m['motion'], m['train_ids_sha1'], m['baked_ids']) == (OFF, ENC, SHA_DOTA, len(ids_all)), f'bake: {m}'
L_BAKE = stage(BAKE, LOCAL / 'stage' / 'bake', ids_all)
L_STATS = stage(BAKE, LOCAL / 'stage' / 'dota_stats', [])          # manifest + statistics only

# Gate R2: the same clips baked with Nexar's statistics, restated to DoTA's == the direct bake
L_NSTATS = stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', [])
nex, gate = LOCAL / 'gate_nexar', LOCAL / 'gate_restat'
for d in (nex, gate):
    shutil.rmtree(d, ignore_errors=True)
run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', L_NSTATS, '--clip-dir', L_X, '--video-dir', L_U,
     '--ids-file', ids_file, '--stride', 1, '--out-dir', nex], log=LOCAL / 'bake.log')
run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'restat', '--from-dir', nex, '--stats-dir', L_STATS,
     '--ids-file', ids_file, '--out-dir', gate], log=LOCAL / 'bake.log')
old, new = load_stats(L_NSTATS), load_stats(L_STATS)
D_V, worst, clip_bad = new.m_u.shape[0], 0.0, []
for v in ids_all:
    a, b, n = np.load(gate / f'{v}.npy'), np.load(L_BAKE / f'{v}.npy'), np.load(nex / f'{v}.npy')
    assert a.shape == b.shape == n.shape, (v, a.shape, b.shape, n.shape)
    worst = max(worst, float(np.max(np.abs(a.astype(np.float64) - b))))
    if not np.array_equal(b[:, :-D_V], n[:, :-D_V]):
        clip_bad.append(v)
assert not clip_bad, f'CLIP columns differ between the two bakes: {clip_bad[:5]}'
assert worst < constants.V2_RESTAT_GATE_ATOL, f'gate R2 FAILED: max |diff| {worst:.2e} -- paste to Claude'
ratio = new.sigma_u / old.sigma_u
STATS_LINE = (f'c {old.c:.4f} (Nexar) -> {new.c:.4f} (DoTA); sigma_u DoTA / Nexar median {np.median(ratio):.3f} '
              f'[p10 {np.percentile(ratio, 10):.3f}, p90 {np.percentile(ratio, 90):.3f}]; '
              f'|m_u diff| / sigma_u(Nexar) median {np.median(np.abs(new.m_u - old.m_u) / old.sigma_u):.3f}')
print(f'gate R2 OK (max |diff| {worst:.2e}) | {STATS_LINE}')

ZS_IDS = {'dada': DADA_IDS, 'dota': CAP_IDS, 'tad': TAD_IDS, DAD_SET: DAD_IDS}
for s, src_dir in ZS_SRC.items():
    dst, ids = ZS_DST[s], ZS_IDS[s]
    if not ((dst / constants.V2_INPUT_MANIFEST_FILENAME).exists() and set(ids) <= on_drive(dst)):
        local = LOCAL / f'restat_{s}'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'restat', '--from-dir', stage(src_dir, LOCAL / 'stage' / f'zs_{s}_nex', ids),
             '--stats-dir', L_STATS, '--ids-file', write_ids(LOCAL / 'ids' / f'zs_{s}.txt', ids), '--out-dir', local], log=LOCAL / f'restat_{s}.log')
        publish(local, dst)
        flush_drive()
    mm, ms = manifest(dst), manifest(src_dir)
    assert (mm['crn'], mm['motion'], mm['train_ids_sha1'], mm['baked_ids']) == (OFF, ENC, SHA_DOTA, len(ids)), f'{s}: {mm}'
    assert mm['stride_over_clip_dir'] == ms.get('stride_over_clip_dir'), f'{s}: stride lost in restat'
    print(f'{s}: {len(ids)} items -> {dst.name}')
L_DATA = stage(DATA, LOCAL / 'stage' / 'data')
assert (DATA / constants.DEFS_FILENAME).read_text() == (WINDOW / constants.DEFS_FILENAME).read_text(), 'defs differ from Nexar_window'
if not KNN.exists():
    local = LOCAL / 'knn' / KNN.name
    run([sys.executable, '-m', 'core.data.knn_cache', '--data-dir', L_DATA, '--dataset', constants.DOTA_TRAIN_DATASET,
         '--clip-dir', L_X, '--output', local], log=LOCAL / 'knn.log')
    publish(local, KNN)
L_KNN = stage(KNN, LOCAL / 'stage' / 'knn' / KNN.name)
print(f"corpus {info} | plan {PLAN}")

## 8. Train **K5dfiDoTA** (§55 DC4)

**Watch the log: past 2 s/step, stop and paste it.** Guards: J10 at 1,764, config diff to K5dfi = {data.dataset, data.frame_stride, train.num_epochs}, last-epoch `mul_mil` > 0.01.

In [ ]:
SETS = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
        f'data.dataset={constants.DOTA_TRAIN_DATASET}', f'data.frame_stride={constants.DOTA_TRAIN_STRIDE}',
        f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}', 'kip.enabled=false', f'loss.dvs_anchor_mode={IGN}',
        'data.is_egocentric=false', f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}',
        f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
        'kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true', 'kim.seam_mask=true',
        'dvs.theta=0.7', 'dvs.theta_ego=0.85']   # = K5dfi's command


def last_epoch(rows, key):
    last = [r for r in rows if r['epoch'] == rows[-1]['epoch']]
    return sum(r[key] for r in last) / len(last)


def check_run(row):
    run_dir, ckpt, _ = row_paths(row)
    mrows = metrics_rows(run_dir / 'metrics.jsonl')
    assert ckpt_step(ckpt) == mrows[-1]['global_step'] == STEPS, f'{row}: J10'
    diff = config_diff(ctrl_cfg, _flat(yaml.safe_load((run_dir / 'config.yaml').read_text())))
    assert set(diff) == {'data.dataset', 'data.frame_stride', 'train.num_epochs'}, f'{row} vs {CTRL}: {diff} (§55 DC4)'
    mul = last_epoch(mrows, 'mul_mil')
    assert mul > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{row}: last-epoch mul_mil {mul:.4f} -- paste to Claude'
    ref = metrics_rows(CTRL_RUN / 'metrics.jsonl')
    print(f'{row}: diff {diff}')
    for key in ('mil', 'mul_mil', 'kim_mil'):
        print(f'  last-epoch {key} {last_epoch(mrows, key):.4f} ({CTRL} {last_epoch(ref, key):.4f})')


def train_row(row):
    run_dir, ckpt, _ = row_paths(row)
    if is_verified(run_dir, STEPS):
        print(f'{row} s{SEED}: verified on Drive -- not retrained')
        check_run(row)
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / row
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    (local / 'env.json').write_text(json.dumps({'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': torch.version.cuda}))
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in SETS for x in ('--set', s)], '--data-dir', L_DATA,
         '--clip-dir', L_BAKE, '--output-dir', local, '--knn-cache', L_KNN, *resume], log=LOCAL / f'train_{row}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{row}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml', 'env.json'):
        publish(local / name, run_dir / name)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{row}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', run_dir / 'verified.json')
    print(f'{row}: trained in {(time.time() - t0) / 60:.0f} min')
    check_run(row)


train_row(ROW)

## 9. Reads: DoTA-CAP-dev (in-domain), DADA, TAD, DAD, Nexar val — K5dfi's protocols, on the bakes with DoTA's statistics

In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_TAD = LOCAL / 'TAD_meta'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every Nexar row read with
    shutil.copy2(f, L_DEFS / f.name)


def read_evaluate(row, name, dataset, data_dir, bake, ids, out, ckpt):
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / row / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', ckpt, '--set', f'data.dataset={dataset}',
             '--data-dir', data_dir, '--clip-dir', stage(bake, LOCAL / 'stage' / f'{name}_bake', ids),
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'{row}_{name}.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == len(ids), r
    print(f"{row} {name}: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")


def read_zero_shot(row):
    _, ckpt, ev = row_paths(row)
    out = ev / 'dota' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt,
             '--input-dir', stage(ZS_DST['dota'], LOCAL / 'stage' / 'cap', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS),
             '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels'), '--split', CAP_DEV,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{row}_dota.log')
        publish(local, out)
    print(f"{row} DoTA-CAP-dev: macro {json.loads((out / 'protocol_b_readout.json').read_text())['macro']['mean']:.4f}")
    out = ev / 'dada' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dada'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', ckpt, '--t2-dir', L_T2,
             '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', stage(ZS_DST['dada'], LOCAL / 'stage' / 'dada', DADA_IDS),
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{row}_dada.log')
        publish(local, out)
    print(f"{row} DADA whole: macro {json.loads((out / 'dada_readout.json').read_text())['macro']['mean']:.4f}")
    read_evaluate(row, 'tad', TAD_NAME, L_TAD, ZS_DST['tad'], TAD_IDS, ev / 'tad' / K5 / f's{SEED}', ckpt)
    read_evaluate(row, 'dad', DAD_NAME, stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta'), ZS_DST[DAD_SET], DAD_IDS,
                  ev / DAD_SET / K5 / f's{SEED}', ckpt)
    flush_drive()


read_zero_shot(ROW)

In [ ]:
from core.tools.nexar_eval import ITEM_AUCS_JSON, READOUT_JSON, VIDEO_STATS_JSON  # noqa: E402


def read_nexar(row):
    _, ckpt, ev = row_paths(row)
    out = ev / 'nexar' / row
    done = all((out / f).exists() for f in (READOUT_JSON, ITEM_AUCS_JSON, VIDEO_STATS_JSON)) and \
        len(list((out / 'scores').glob('*.npz'))) == len(NEXAR_IDS)
    if not done:
        L_X = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
        L_U = stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS)
        local = LOCAL / 'eval' / row / 'nexar'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.nexar_eval', '--run', f's{SEED}', ckpt, '--stats-dir', L_STATS,
             '--video-s1-dir', L_U, '--census', CENSUS, '--clip-s1-dir', L_X,
             '--data-dir', L_DEFS, '--split', NEXAR_SPLIT, '--save-scores', '--out-dir', local], log=LOCAL / f'{row}_nexar.log')
        publish(local, out)
        flush_drive()
    r = json.loads((out / READOUT_JSON).read_text())
    assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS), (r['split'], r['videos'])
    assert r['global_steps'] == {f's{SEED}': STEPS} and Path(r['checkpoints'][f's{SEED}']) == ckpt, (r['global_steps'], r['checkpoints'])
    assert r['stats_manifest']['train_ids_sha1'] == SHA_DOTA, 'Nexar val was read with other statistics'
    print(f"{row}: crops macro {r['crops']['mean_over_placements']['macro']['mean']:.4f} | "
          f"whole macro {r['whole']['macro']['mean']:.4f} | clip AUC {r['whole']['clip_auc_max']:.4f}")


read_nexar(ROW)

## 10. K5dfiDoTA vs K5dfi -> **paste `REPORTS/k5dfidota_report.md` to Claude**

In [ ]:
from core.tools import k5_whole_report as kw     # noqa: E402

local = LOCAL / 'report'
shutil.rmtree(local, ignore_errors=True)
local.mkdir(parents=True)
EV = {CTRL: CTRL_EV, ROW: row_paths(ROW)[2], K5F: K5F_EV}
table_args = [a for row, ev in EV.items() for s in ('dota', 'dada', 'tad', DAD_SET) for a in ('--read', s, row, ev / s / K5)]
NAME = constants.KIM_DOTA_TRAIN_TABLE_NAME
run([sys.executable, '-m', 'core.tools.zs_arm_table', '--control', CTRL, '--seed', SEED, *table_args, '--name', NAME, '--out-dir', local])
nx = kw.nexar_set({kw.K5: CTRL_EV / 'nexar' / CTRL, ROW: EV[ROW] / 'nexar' / ROW}, new=ROW)   # the reader keys the control as K5
dl = nx['delta']
env = json.loads((row_paths(ROW)[0] / 'env.json').read_text())
lines = ['# K5dfiDoTA (KIM_RUN.md §55), s2024, n = 1 — printed, decides nothing; K5f stays', '',
         f"Corpus {info}; {STEPS} steps; config diff to K5dfi {{data.dataset, data.frame_stride, train.num_epochs}}; {env['gpu']}.", '',
         f'Input statistics fitted on {len(TRAIN_SRC)} DoTA train clips ({SHA_DOTA[:12]}), used at every read of this row; K5dfi / K5f used Nexar train. '
         f'Gate R2 max |diff| {worst:.2e}. {STATS_LINE}', '',
         'DoTA-CAP-dev is IN-DOMAIN for K5dfiDoTA (zero-shot for K5dfi and K5f). Pixel gate not valid (§52.2); trained by the user\'s decision.', '',
         'Rulers on DoTA-CAP-dev macro: cubic position fitted on the train clips 0.8545 [0.8386, 0.8695] (no pixel); t/N 0.5573. '
         '566 of 569 dev clips share a YouTube id with a train clip.', '',
         '## Nexar val (crops macro, paired per video)', '', '| row | macro crops | macro whole | micro | AP |', '|---|---|---|---|---|']
for row, c in nx['rows'].items():
    name = CTRL if row == kw.K5 else row
    lines.append(f"| {name} | {c['macro_crops']['mean']:.4f} | {c['macro_whole']['mean']:.4f} | {c['micro']:.4f} | {c['ap']:.4f} |")
lines += ['', f"K5dfiDoTA - K5dfi: {dl['mean']:+.4f} [{dl['low']:+.4f}, {dl['high']:+.4f}]", '', '## Four sets (control K5dfi; K5f printed)', '',
          (local / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md').read_text()]
(local / f'{constants.KIM_DOTA_TRAIN_REPORT_STEM}.md').write_text('\n'.join(lines) + '\n')
publish(local, REPORTS)
print((REPORTS / f'{constants.KIM_DOTA_TRAIN_REPORT_STEM}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5dfidota_report.md to Claude')